In [ ]:
import pandas as pd

1. Is there a correlation between graduation rates and number of economically disadvantaged students in each district?

In [ ]:
# Load datasets
grad_pct = pd.read_csv("../data/processed/grad_pct_per_DISTRICT.csv")
per_ecdis = pd.read_csv("../data/processed/PER_ECDIS_per_DISTRICT.csv")

In [ ]:
grad_pct

- Only keep data needed for analysis

Note: graduation rate is not recorded for cohorts of fewer than 5 students

In [ ]:
# Only keep necessary rows
grad_pct = grad_pct.loc[(grad_pct.subgroup_code == 1) &  # Only keep 'All Students' numbers
                        (grad_pct.MEMBERSHIP_CODE == 9.0) & # Only keep 2021 Total Cohort - 4 Year Outcome
                        (grad_pct.GRAD_PCT != '-') # Drop all rows where grad pct is not recorded
                        ] 

# Drop all unnecessary columns
grad_pct = grad_pct.drop(['REPORT_SCHOOL_YEAR', 'AGGREGATION_TYPE', 'MEMBERSHIP_CODE', 'MEMBERSHIP_DESC', 'subgroup_code', 'SUBGROUP_NAME'], axis=1).reset_index(drop=True)

grad_pct

In [ ]:
# Convert metrics to appropriate types
grad_pct = grad_pct.astype({
    'AGGREGATION_CODE': str,
    'ENROLL_CNT': 'int32',
    'GRAD_CNT': 'int32'
})

# Clean GRAD_PCT column
grad_pct['GRAD_PCT'] = (grad_pct['GRAD_CNT'] / grad_pct['ENROLL_CNT']).round(2)

grad_pct

Do the same for per_ecdis dataset

In [ ]:
per_ecdis

In [ ]:
# Only keep year 2025, and drop year column
per_ecdis = per_ecdis.loc[per_ecdis.YEAR == 2025].reset_index(drop=True).drop('YEAR', axis=1)

# Keep percentage values between 0 and 1
per_ecdis['PER_ECDIS'] = per_ecdis['PER_ECDIS'] / 100

# Convert types
per_ecdis = per_ecdis.astype({
    'ENTITY_CD': 'str'
})

In [ ]:
per_ecdis

In [ ]:
master_dataset = grad_pct.merge(per_ecdis, left_on=['AGGREGATION_CODE', 'AGGREGATION_NAME'], right_on=['ENTITY_CD', 'ENTITY_NAME']).drop(['ENTITY_CD', 'ENTITY_NAME'],  axis=1).reset_index(drop=True)

master_dataset

In [ ]:
master_dataset.to_csv('../data/processed/GRAD_PCT_vs_PER_ECDIS.csv', index=False)

2. What are the grade distributions for charter vs non-charter schools in each subject?

In [ ]:
charter_filter = pd.read_csv(r"..\data\processed\INSTITUTION_ID_to_NEEDS_INDEX.csv")
charter_filter

In [ ]:
# Add column for charter vs non-charter school
charter_filter.loc[charter_filter.NEEDS_INDEX == 7, 'SCHOOL_TYPE'] = 'Charter'
charter_filter.loc[charter_filter.NEEDS_INDEX != 7, 'SCHOOL_TYPE'] = 'District'

# Drop all unneeded columns
charter_filter = charter_filter.drop(['YEAR', 'NEEDS_INDEX', 'NEEDS_INDEX_DESCRIPTION'], axis=1)
charter_filter

In [ ]:
per_prof = pd.read_csv(r"..\data\processed\INSTITUTION_ID_per_subject_PER_PROF.csv")
per_prof

In [ ]:
# Remove aggregated rows for NYC Public Schools County
per_prof = per_prof[per_prof.INSTITUTION_ID != 7889678368]

# Drop unnecessary columns
per_prof = per_prof.drop(["YEAR", "SUBGROUP_NAME"], axis=1, errors='ignore')

# Only keep certain subjects
per_prof = per_prof.loc[per_prof.SUBJECT.isin(['Regents NF Global History', "Regents US History&Gov't", 'Regents Common Core English Language Art', 'Regents Phy Set/Physics', 'Regents Phy Set/Chemistry', 'Regents Life Science: Biology', 'Regents Common Core Geometry', 'Regents Common Core Algebra II', 'Regents Earth and Space Sciences'])]

# Convert metrics to appropriate types
per_prof = per_prof.astype({
    'TESTED': 'int32'
})

per_prof

In [ ]:
per_prof.loc[per_prof.PER_PROF == 's', 'TESTED'].drop_duplicates()

In [ ]:
per_prof.SUBJECT.drop_duplicates().tolist()